# 28. Finer Refined Comparable

27번이 public에서 2367.40291로 개선되었으므로, 27 주변만 더 좁게 미세 탐색합니다.

- Scope: Dong 고정
- Gate: low_range_q50 고정
- HistMinAbs: 25 고정
- 27 주변 Prior/TopK/Alpha/Cap만 탐색
- OOF 기준 Fold2/Fold3 모두 안정적으로 개선될 때만 제출 파일 생성


In [1]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 220)
pd.set_option('display.float_format', lambda x: f'{x:,.6f}')

candidates = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next((p for p in candidates if (p / 'data').exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('프로젝트 루트를 찾을 수 없습니다.')

# 27을 재현합니다. 27 내부는 25/24/17을 모두 leakage-safe 방식으로 재현합니다.
nb27 = json.loads((PROJECT_ROOT / 'notebooks' / '27_fine_refined_comparable.ipynb').read_text())
code27 = ''.join(nb27['cells'][1]['source'])
ns27 = {}
exec(code27, ns27)

rmse = ns27['rmse']
oof = ns27['oof'].copy()
folds = ns27['folds']
y_all = ns27['y_all']
pred17_all = ns27['pred17_all']
pred15_all = ns27['pred15_all']
hist_gap_all = ns27['hist_gap_all']
comparable_predict = ns27['comparable_predict']
train = ns27['train']
inner15 = ns27['inner15']
sample_submission = ns27['sample_submission']
SUBMISSION_DIR = ns27['SUBMISSION_DIR']
ns25 = ns27['ns25']
ns24 = ns25['ns24']

BASELINE_27 = {
    'scope': 'dong', 'prior': 100, 'top_k': 30,
    'gate': 'low_range_q50', 'alpha': 0.105, 'cap': 2800, 'hist_min_abs': 25,
}
SCOPE = 'dong'
GATE_NAME, GATE_COL, GATE_Q = ('low_range_q50', 'pred_range', 0.50)
HIST_MIN_ABS = 25
PRIORS = [90, 100, 110]
TOP_KS = [25, 30, 35]
ALPHAS = [0.100, 0.105, 0.110, 0.115]
CAPS = [2600, 2800, 3000, 3200]

base17_rmse = rmse(y_all, pred17_all)

def make_oof_comparable(scope, prior, top_k):
    fold_preds = []
    for fold in folds:
        valid_months = inner15['TIME_FOLDS'][fold - 1][1]
        valid_df = train.loc[train['Transaction_YearMonth'].isin(valid_months)].copy()
        fit_df = train.loc[train['Transaction_YearMonth'] < valid_months.min()].copy()
        pred, _ = comparable_predict(fit_df, valid_df, scope=scope, prior=prior, top_k=top_k)
        fold_preds.append(pred)
    return np.concatenate(fold_preds)

comp_cache = {}
for prior in PRIORS:
    for top_k in TOP_KS:
        comp_cache[(prior, top_k)] = make_oof_comparable(SCOPE, prior, top_k)

rows = []
detail_tables = {}
pred_tables = {}
for (prior, top_k), comp_pred in comp_cache.items():
    comp_gap_all = comp_pred - pred17_all
    for alpha in ALPHAS:
        for cap in CAPS:
            pred28 = pred17_all.copy()
            detail = []
            offset = 0
            for fold in folds:
                frame = oof.loc[oof['Fold'] == fold].copy()
                n = len(frame)
                y = frame['Target'].to_numpy()
                pred17 = frame['Pred17'].to_numpy()
                comp_gap = comp_gap_all[offset:offset+n]
                hist_gap = hist_gap_all[offset:offset+n]
                if fold == 1:
                    mask = np.zeros(n, dtype=bool)
                else:
                    agree = (comp_gap * hist_gap) > 0
                    strong = np.abs(hist_gap) >= HIST_MIN_ABS
                    bounded = np.abs(comp_gap) <= cap * 6
                    fit_hist = oof.loc[oof['Fold'] < fold].copy()
                    threshold = fit_hist[GATE_COL].quantile(GATE_Q)
                    gate_mask = frame[GATE_COL].to_numpy() <= threshold
                    mask = agree & strong & bounded & gate_mask
                clipped_gap = np.clip(comp_gap, -cap, cap)
                pred_fold = pred17.copy()
                pred_fold[mask] = np.clip(pred17[mask] + alpha * clipped_gap[mask], 0, None)
                pred28[offset:offset+n] = pred_fold
                detail.append({
                    'Fold': int(fold),
                    'Rows': n,
                    '17_RMSE': rmse(y, pred17),
                    '28_RMSE': rmse(y, pred_fold),
                    '28_vs_17_Improvement': rmse(y, pred17) - rmse(y, pred_fold),
                    'Applied_Rows': int(mask.sum()),
                    'Mean_abs_move': float(np.mean(np.abs(pred_fold - pred17))),
                })
                offset += n
            detail = pd.DataFrame(detail)
            eligible = detail.loc[detail['Fold'] > 1]
            key = (prior, top_k, alpha, cap)
            detail_tables[key] = detail
            pred_tables[key] = pred28
            rows.append({
                'Scope': SCOPE,
                'Prior': prior,
                'TopK': top_k,
                'Gate': GATE_NAME,
                'Alpha': alpha,
                'Cap': cap,
                'HistMinAbs': HIST_MIN_ABS,
                '17_Pooled_RMSE': base17_rmse,
                '28_Pooled_RMSE': rmse(y_all, pred28),
                'Pooled_Improvement': base17_rmse - rmse(y_all, pred28),
                'Improved_Eligible_Folds': int((eligible['28_vs_17_Improvement'] > 0).sum()),
                'Worst_Eligible_Improvement': eligible['28_vs_17_Improvement'].min(),
                'Fold2_Improvement': detail.loc[detail['Fold'] == 2, '28_vs_17_Improvement'].iloc[0],
                'Fold3_Improvement': detail.loc[detail['Fold'] == 3, '28_vs_17_Improvement'].iloc[0],
                'Mean_abs_move': detail['Mean_abs_move'].mean(),
                'Applied_Rows_F2': detail.loc[detail['Fold'] == 2, 'Applied_Rows'].iloc[0],
                'Applied_Rows_F3': detail.loc[detail['Fold'] == 3, 'Applied_Rows'].iloc[0],
                'Is_27_Config': (
                    prior == BASELINE_27['prior'] and top_k == BASELINE_27['top_k'] and
                    abs(alpha - BASELINE_27['alpha']) < 1e-12 and cap == BASELINE_27['cap']
                ),
            })

summary = pd.DataFrame(rows).sort_values(
    ['Pooled_Improvement', 'Worst_Eligible_Improvement', 'Fold3_Improvement'],
    ascending=[False, False, False],
).reset_index(drop=True)
display(summary.head(60))

ref27 = summary.loc[summary['Is_27_Config']].iloc[0]
print('Reproduced 27 config:')
display(ref27.to_frame('value'))
display(detail_tables[(BASELINE_27['prior'], BASELINE_27['top_k'], BASELINE_27['alpha'], BASELINE_27['cap'])])

stable = summary.query(
    'Improved_Eligible_Folds == 2 and Worst_Eligible_Improvement > 2.5 and Fold3_Improvement > 2.5 and Applied_Rows_F2 >= 30 and Applied_Rows_F2 <= 45 and Applied_Rows_F3 >= 35 and Applied_Rows_F3 <= 50 and Mean_abs_move < 35'
).copy()

# 27보다 OOF가 의미 있게 낫거나, 유사하면서 Fold3가 더 좋은 후보만 허용합니다.
stable = stable.loc[
    (stable['Pooled_Improvement'] >= ref27['Pooled_Improvement'] + 0.05) |
    ((stable['Pooled_Improvement'] >= ref27['Pooled_Improvement'] - 0.10) & (stable['Fold3_Improvement'] >= ref27['Fold3_Improvement'] + 0.20))
].copy()

if len(stable) == 0:
    print('27을 대체할 finer comparable 후보가 없습니다. 제출 파일을 생성하지 않습니다.')
    best = ref27.copy()
    CREATE_SUBMISSION = False
else:
    stable['Distance_From_27'] = (
        (stable['Prior'] - BASELINE_27['prior']).abs() / 100
        + (stable['TopK'] - BASELINE_27['top_k']).abs() / 100
        + (stable['Alpha'] - BASELINE_27['alpha']).abs() * 10
        + (stable['Cap'] - BASELINE_27['cap']).abs() / 2000
    )
    stable['OOF_Rank_Bucket'] = stable['Pooled_Improvement'].max() - stable['Pooled_Improvement'] <= 0.15
    pool = stable.loc[stable['OOF_Rank_Bucket']].copy() if stable['OOF_Rank_Bucket'].any() else stable.copy()
    best = pool.sort_values(['Distance_From_27', 'Pooled_Improvement'], ascending=[True, False]).iloc[0].copy()
    CREATE_SUBMISSION = True
    print('선택된 28 finer refined comparable correction:')

display(best.to_frame('value'))
best_key = (int(best['Prior']), int(best['TopK']), float(best['Alpha']), int(best['Cap']))
display(detail_tables[best_key])

leakage_audit = pd.Series({
    '27 baseline reproduced from leakage-safe notebook': True,
    'OOF comparable pools use only transactions earlier than validation months': True,
    'Agreement gate uses OOF Pred15/Pred17 only, not validation target': True,
    'Gate thresholds computed from earlier OOF folds only': True,
    'Finer refinement selected from Train OOF only': True,
    'No Test distribution/statistics used for selection': True,
    'Final Test comparable prediction uses full Train only after selection': True,
})
display(leakage_audit.rename('Passed'))
assert leakage_audit.all()

if CREATE_SUBMISSION:
    prior = int(best['Prior'])
    top_k = int(best['TopK'])
    alpha = float(best['Alpha'])
    cap = int(best['Cap'])
    comp_test, _ = comparable_predict(train, ns25['ns24']['test'], scope=SCOPE, prior=prior, top_k=top_k)
    comp_gap_test = comp_test - ns25['ns24']['test_pred17']
    hist_gap_test = ns25['ns24']['test_pred17'] - ns25['inner15']['test_pred']
    agree_test = (comp_gap_test * hist_gap_test) > 0
    strong_test = np.abs(hist_gap_test) >= HIST_MIN_ABS
    bounded_test = np.abs(comp_gap_test) <= cap * 6
    threshold = oof[GATE_COL].quantile(GATE_Q)
    gate_mask_test = ns25['ns24']['test_frame'][GATE_COL].to_numpy() <= threshold
    mask_test = agree_test & strong_test & bounded_test & gate_mask_test
    clipped_gap_test = np.clip(comp_gap_test, -cap, cap)
    test_pred28 = ns25['ns24']['test_pred17'].copy()
    test_pred28[mask_test] = np.clip(ns25['ns24']['test_pred17'][mask_test] + alpha * clipped_gap_test[mask_test], 0, None)

    prediction_frame = pd.DataFrame({'ID': ns25['ns24']['test']['ID'], 'Target': test_pred28})
    submission = sample_submission[['ID']].merge(prediction_frame, on='ID', how='left', validate='one_to_one')
    assert submission['Target'].notna().all()
    assert len(submission) == len(sample_submission)
    assert sample_submission['ID'].equals(submission['ID'])
    output_path = SUBMISSION_DIR / '28_finer_refined_comparable_submission.csv'
    submission.to_csv(output_path, index=False)
    print(f'Saved: {output_path}')
    display(submission.head())
else:
    output_path = None
    print('No submission saved for 28. Keep submitting 27_fine_refined_comparable_submission.csv.')

result = {
    'create_submission': CREATE_SUBMISSION,
    'selected_prior': None if not CREATE_SUBMISSION else int(best['Prior']),
    'selected_top_k': None if not CREATE_SUBMISSION else int(best['TopK']),
    'selected_alpha': None if not CREATE_SUBMISSION else float(best['Alpha']),
    'selected_cap': None if not CREATE_SUBMISSION else int(best['Cap']),
    'selected_oof_improvement': float(best['Pooled_Improvement']),
    'reference_27_oof_improvement': float(ref27['Pooled_Improvement']),
    'submission_path': None if output_path is None else str(output_path),
}
print(result)


Train: (1969, 11)
Train period: 202401 ~ 202512


,Fold,Train_rows,Valid_rows,04_RMSE,05_RMSE,06_RMSE,08_RMSE,09_RMSE,09_RMSLE,08_vs_06_Improvement,09_vs_08_Improvement,Local_Applied_Rows,Second_Local_Applied_Rows,Third_Local_Applied_Rows
0,Train-derived Fold 1,1201,278,"2,107.743326","2,107.743326","2,107.743326","2,107.743326","2,107.743326",0.055311,0.000000,0.000000,0,0,0
1,Train-derived Fold 2,1479,244,"2,634.763161","2,628.945780","2,621.229339","2,618.051538","2,614.984309",0.080279,3.177802,3.067229,141,103,108
2,Train-derived Fold 3,1723,246,"2,525.601387","2,522.504744","2,522.382035","2,520.144861","2,516.302312",0.071534,2.237174,3.842550,146,100,100


04 pooled RMSE: 2,420.08
05 pooled RMSE: 2,417.04
06 pooled RMSE: 2,414.33
08 pooled RMSE: 2,412.49
09 pooled RMSE: 2,410.15
Local-corrected improved folds: 2/2
Second-local improved folds: 2/2
Third-local improved folds: 2/2
08 vs 06 pooled improvement: 1.84
09 vs 08 pooled improvement: 2.34


Imputation statistics fitted on Train/fold-fit only                  True
OneHotEncoder fitted on Train/fold-fit only                          True
No independent dummy encoding on Test                                True
Fixed bins avoid validation/test distribution fitting                True
Target encoding excludes each training row target                    True
Residual model trained from time-OOF residuals only                  True
Local residual maps fitted from previous OOF residuals only          True
Second local correction selected with strict Train OOF thresholds    True
Third local correction selected with strict Train OOF thresholds     True
Validation uses strictly earlier transactions                        True
Weights, shrinkage and local corrections selected without Test       True
Test limited to transform and predict                                True
Name: Passed, dtype: bool

Saved: /Users/joyeojin/10_Dev/11_Projects/korea-apartment-price-prediction/submissions/09_residual08_cluster_correction_submission.csv


...

,Fold,Rows,17_RMSE,28_RMSE,28_vs_17_Improvement,Applied_Rows,Mean_abs_move
0,1,278,"2,107.743326","2,107.743326",0.000000,0,0.000000
1,2,244,"2,588.127758","2,583.858593",4.269165,37,39.486095
2,3,246,"2,511.258274","2,506.635060",4.623215,41,41.748513


27 baseline reproduced from leakage-safe notebook                            True
OOF comparable pools use only transactions earlier than validation months    True
Agreement gate uses OOF Pred15/Pred17 only, not validation target            True
Gate thresholds computed from earlier OOF folds only                         True
Finer refinement selected from Train OOF only                                True
No Test distribution/statistics used for selection                           True
Final Test comparable prediction uses full Train only after selection        True
Name: Passed, dtype: bool

Saved: /Users/joyeojin/10_Dev/11_Projects/korea-apartment-price-prediction/submissions/28_finer_refined_comparable_submission.csv


,ID,Target
0,TR_2427,"36,937.484718"
1,TR_0028,"48,170.602450"
2,TR_1461,"29,323.441340"
3,TR_1977,"47,532.510021"
4,TR_2351,"46,969.224666"


{'create_submission': True, 'selected_prior': 100, 'selected_top_k': 30, 'selected_alpha': 0.115, 'selected_cap': 3000, 'selected_oof_improvement': 3.0124214234633655, 'reference_27_oof_improvement': 2.775609094767333, 'submission_path': '/Users/joyeojin/10_Dev/11_Projects/korea-apartment-price-prediction/submissions/28_finer_refined_comparable_submission.csv'}
